# Processing High-Dimensional Data with 200+ Columns for ML Feature Selection

When dealing with datasets with 200+ columns (features), proper feature selection and dimensionality reduction become crucial for building effective machine learning models. Here's a comprehensive approach with a real-world example.

## Example Dataset: Human Activity Recognition (HAR) with Smartphones

Let's use the [UCI HAR Dataset](https://archive.ics.uci.edu/ml/datasets/human+activity+recognition+using+smartphones) which contains 561 features derived from smartphone sensor data to classify human activities (walking, sitting, standing, etc.).



### Step 1: Imports

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.ensemble import RandomForestClassifier
import matplotlib.pyplot as plt
import seaborn as sns

### Step 2 : Loading data

In [ ]:
# Load feature names (561 columns)
feature_names = pd.read_csv('UCI_HAR_Dataset/features.txt', sep='\s+', header=None, names=['index', 'name'])

# Load training and test data
X_train = pd.read_csv('UCI_HAR_Dataset/train/X_train.txt', sep='\s+', header=None)
X_test = pd.read_csv('UCI_HAR_Dataset/test/X_test.txt', sep='\s+', header=None)

# Load labels
y_train = pd.read_csv('UCI_HAR_Dataset/train/y_train.txt', sep='\s+', header=None, names=['label'])
y_test = pd.read_csv('UCI_HAR_Dataset/test/y_test.txt', sep='\s+', header=None, names=['label'])

# Combine train and test for feature selection (optional)
X = pd.concat([X_train, X_test])
y = pd.concat([y_train, y_test])

# Set feature names
X.columns = feature_names['name']

print(f"Data shape: {X.shape}")  # Should be (10299, 561)
print(f"Sample features:\n{X.iloc[0, :5]}")  # Show first 5 features of first sample

In [ ]:
# Standardize the data
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [ ]:
# Handle missing values if any
X_scaled = pd.DataFrame(X_scaled).fillna(0).values

### Step 3: Correlation Analysis


In [ ]:
# Calculate correlation matrix (for a subset if memory is constrained)
corr_matrix = pd.DataFrame(X_scaled).iloc[:, :100].corr().abs()  # First 100 features

In [ ]:
# Plot correlation heatmap
plt.figure(figsize=(12,10))
sns.heatmap(corr_matrix, cmap='viridis')
plt.title('Feature Correlation Heatmap')
plt.show()


In [ ]:
# Remove highly correlated features
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
to_drop = [column for column in upper.columns if any(upper[column] > 0.95)]
print(f"Dropping {len(to_drop)} highly correlated features")
X_reduced = pd.DataFrame(X_scaled).drop(columns=to_drop)

In [ ]:
# Select top k features using ANOVA F-value
selector = SelectKBest(f_classif, k=50)
X_new = selector.fit_transform(X_reduced, y)


### Step 4: Univariate Feature Selection

In [ ]:
# Get selected feature indices
selected_features = selector.get_support(indices=True)
feature_scores = selector.scores_[selected_features]

In [ ]:
# Plot feature importance
plt.figure(figsize=(10,6))
plt.bar(range(len(feature_scores)), feature_scores)
plt.title("Univariate Feature Scores (ANOVA F-value)")
plt.xlabel("Feature Index")
plt.ylabel("F-score")
plt.show()


### Step 5: Model-Based Feature Selection

In [ ]:
# Use Random Forest to get feature importance
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_scaled, y)

In [ ]:
# Get feature importances
importances = rf.feature_importances_
indices = np.argsort(importances)[::-1]

In [ ]:
# Plot top 30 features
plt.figure(figsize=(10,6))
plt.title("Random Forest Feature Importance")
plt.bar(range(30), importances[indices[:30]])
plt.xticks(range(30), indices[:30], rotation=90)
plt.show()

In [ ]:

# Select top features based on importance
threshold = 0.01  # Only keep features with importance > 1%
important_features = np.where(importances > threshold)[0]
X_important = X_scaled[:, important_features]

### Step 6: Dimensionality Reduction with PCA

In [ ]:
# Apply PCA to further reduce dimensions
pca = PCA(n_components=0.95)  # Keep 95% of variance
X_pca = pca.fit_transform(X_scaled)

print(f"Reduced from {X_scaled.shape[1]} to {X_pca.shape[1]} dimensions")

In [ ]:
# Plot explained variance
plt.figure(figsize=(8,5))
plt.plot(np.cumsum(pca.explained_variance_ratio_))
plt.xlabel('Number of Components')
plt.ylabel('Cumulative Explained Variance')
plt.show()


### Step 7: Recursive Feature Elimination (RFE)

In [ ]:
from sklearn.feature_selection import RFECV
from sklearn.linear_model import LogisticRegression

# Use a simpler model for RFE
estimator = LogisticRegression(max_iter=10)
#selector = RFECV(estimator, step=1, cv=5)
selector = selector.fit(X_scaled, y)

In [ ]:
# Get optimal number of features
print(f"Optimal number of features: {selector.n_features_}")

In [ ]:

# Transform the data
X_rfe = selector.transform(X_scaled)


### Step 8: Final Feature Set Evaluation

In [ ]:
from sklearn.model_selection import cross_val_score

# Compare performance with different feature sets
feature_sets = {
    'Original': X_scaled,
    'Correlation Reduced': X_reduced,
    'Univariate Selection': X_new,
    'RF Importance': X_important,
    'PCA': X_pca,
    'RFE': X_rfe
}

for name, X_set in feature_sets.items():
    scores = cross_val_score(rf, X_set, y, cv=5, scoring='accuracy')
    print(f"{name}: Mean Accuracy = {scores.mean():.3f} ± {scores.std():.3f}")




## Alternative Approaches for Very High-Dimensional Data

For datasets with even more features (e.g., genomics data with 50,000+ features):

1. **Autoencoders**: Neural networks for unsupervised dimensionality reduction
2. **t-SNE/UMAP**: For visualization and potential feature extraction
3. **Feature Agglomeration**: Cluster similar features together
4. **Sparse PCA**: For high-dimensional data where regular PCA fails
5. **Stability Selection**: Repeated feature selection under data perturbations



## Real-World Example: Gene Expression Data

The [TCGA Pan-Cancer Atlas](https://www.cancer.gov/tcga) dataset contains gene expression data with ~20,000 genes (features) for cancer classification.

In [ ]:
# # Example processing pipeline for gene data
# from sklearn.linear_model import LassoCV

# # Use Lasso for feature selection (sparse solution)
# lasso = LassoCV(cv=5, max_iter=10000)
# lasso.fit(X_scaled, y)

# # Get non-zero coefficients
# selected_genes = np.where(lasso.coef_ != 0)[0]
# X_genes = X_scaled[:, selected_genes]
# print(f"Selected {len(selected_genes)} genes from {X_scaled.shape[1]} initial features")



## Key Takeaways

1. **Start with correlation analysis** to remove redundant features
2. **Use univariate methods** for quick initial feature screening
3. **Apply model-based selection** for more sophisticated selection
4. **Consider dimensionality reduction** when feature interactions are important
5. **Validate performance** with cross-validation for each feature set
6. **Balance computational cost** with potential accuracy gains
7. **Domain knowledge** should guide the process where possible

The optimal approach depends on your specific dataset size, computational resources, and the nature of your machine learning task.